# Tome C2, Chapter 1: Measuring Algorithms (Python companion)

Exercises first, solutions at the end. We solve recurrences exactly and watch them approach the predictions of the master theorem, run the amortised analyses of the chapter as cost traces, make the Horn-clause solver of Theorem 1.13 linear and time it, play the adversaries of Theorems 1.18 and 1.19 against real algorithms, implement Ford and Johnson's merge insertion and compare it with the information bound $\lceil\log_2 n!\rceil$, and count Euclid's algorithm in divisions and in bits.

*Before each solution, predict what the output will look like.*

In [ ]:
import itertools, random, math, time
from functools import lru_cache
import numpy as np
import matplotlib.pyplot as plt
random.seed(1)

## Exercises

**P1.1 (★) Recurrences.** Compute $T(n)$ exactly for $T(n)=aT(n/b)+f(n)$ on powers of $b$ for the examples of Example 1.6 and Exercise 1.1, divide by the prediction of Theorem 1.5, and check that the ratio settles. Then do the same with floors and ceilings for all $n$ (Proposition 1.7), and for the lopsided $T(n)=T(\lfloor n/3\rfloor)+T(\lfloor 2n/3\rfloor)+n$ of Proposition 1.9. Finally, watch $n\log\log n$ appear in Exercise 1.2.

**P1.2 (★) Merge sort.** Count the comparisons of merge sort: the worst case for each $n\le 64$ from the recurrence, the bound $n\lceil\log_2n\rceil$ of Proposition 1.8, the lower bound $\lceil\log_2n!\rceil$ of Theorem 1.15, and the average over random inputs.

**P1.3 (★) Amortised cost traces.** Trace the total cost of $m$ operations for the binary counter (Proposition 1.11), the dynamic array (Proposition 1.12), the two shrinking rules of Exercise 1.6 under an adversarial sequence, and the queue made of two stacks (Exercise 1.5). Then look at how CPython actually grows a list.

**P1.4 (★★) Horn clauses in linear time.** Implement the counter algorithm of Theorem 1.13 and the naive loop of Tome C1, Theorem 1.16. Check them against each other and against brute force on small random Horn formulas, and time both as the formula grows.

**P1.5 (★★) Adversaries.** Implement the adversary of Theorem 1.18 and run several max-and-min algorithms against it: none should finish in fewer than $\lceil 3n/2\rceil-2$ comparisons. Do the same with the weight adversary of Theorem 1.19 and second-largest algorithms, and count the matches of Lewis Carroll's corrected tournament.

**P1.6 (★★★) Merge insertion.** Implement Ford and Johnson's merge insertion. Find its worst case over all orderings for $n\le 8$, compare with the formula $\sum_{k\le n}\lceil\log_2(3k/4)\rceil$ and with $\lceil\log_2n!\rceil$, and check Exercise 1.8 (five elements in seven comparisons).

**P1.7 (★) Euclid, counted.** Find, for each number of digits, the pair $a>b$ needing the most divisions and check that it is a pair of consecutive Fibonacci numbers (Lamé). Then count the bit operations of schoolbook division along Euclid's algorithm and compare with $\ell(a)\ell(b)$ (Proposition 1.3).

## Solutions

### P1.1 Recurrences

`solve(a, b, f, K)` returns $T(b^k)$ for $k\le K$ by iterating the recurrence with $T(1)=1$. For each example the last column is $T(n)$ divided by the prediction.

In [ ]:
def solve(a, b, f, K):
    T = [1.0]
    for k in range(1, K + 1):
        n = b ** k
        T.append(a * T[-1] + f(n))
    return T

lg = math.log2
examples = [
    ("merge sort 2T(n/2)+n",        2, 2, lambda n: n,          lambda n: n * lg(n)),
    ("binary search T(n/2)+1",      1, 2, lambda n: 1,          lambda n: lg(n)),
    ("Karatsuba 3T(n/2)+n",         3, 2, lambda n: n,          lambda n: n ** lg(3)),
    ("Strassen 7T(n/2)+n^2",        7, 2, lambda n: n * n,      lambda n: n ** lg(7)),
    ("root wins 2T(n/2)+n^2",       2, 2, lambda n: n * n,      lambda n: n * n),
    ("4T(n/2)+n^2",                 4, 2, lambda n: n * n,      lambda n: n * n * lg(n)),
    ("9T(n/3)+n",                   9, 3, lambda n: n,          lambda n: n * n),
    ("2T(n/4)+sqrt n",              2, 4, lambda n: n ** 0.5,   lambda n: n ** 0.5 * lg(n)),
]
for name, a, b, f, g in examples:
    K = 30 if b == 2 else 18
    T = solve(a, b, f, K)
    ratios = [T[k] / g(b ** k) for k in (K // 3, 2 * K // 3, K)]
    print(f"{name:28s} ratio T/prediction at k={K//3},{2*K//3},{K}: " + "  ".join(f"{r:8.4f}" for r in ratios))

The ratios settle to constants: that is what $\Theta$ means. In case (b) the convergence is slow (the ratio for merge sort is $1+1/\log_2 n$), which is why $\Theta$ hides constants *and* lower-order terms.

Now all $n$, with $\lceil n/2\rceil$ and $\lfloor n/2\rfloor$ (Proposition 1.7), and the lopsided recurrence of Proposition 1.9, whose bound is $n\log_2n/h$ with $h=\log_23-2/3$.

In [ ]:
N = 1 << 16
T = [0, 0] + [0] * (N - 1)
for n in range(2, N + 1):
    T[n] = T[(n + 1) // 2] + T[n // 2] + n - 1          # merge sort worst case
U = [0, 1] + [0] * (N - 1)
for n in range(2, N + 1):
    U[n] = U[n // 3] + U[2 * n // 3] + n                # lopsided
h = lg(3) - 2 / 3
xs = np.arange(2, N + 1)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(xs, np.array(T[2:]) / (xs * np.log2(xs)), lw=0.6)
ax[0].set_title(r"merge sort: $C(n)/(n\log_2 n)$, all $n$"); ax[0].set_xscale("log")
ax[1].plot(xs, np.array(U[2:]) / (xs * np.log2(xs) / h), lw=0.6, color="C1")
ax[1].set_title(r"$T(n/3)+T(2n/3)+n$ over $n\log_2 n/h$"); ax[1].set_xscale("log")
plt.tight_layout(); plt.show()
print("max ratio for the lopsided recurrence:", max(U[n] / (n * lg(n) / h) for n in range(2, N + 1)))

The merge-sort ratio wiggles between powers of $2$ but stays below $1$ (Proposition 1.8). The lopsided ratio stays bounded, as Proposition 1.9 promises; it exceeds $1$ for small $n$, where the base cases weigh, which is why the constant there is $\max(C_0,c/h)$ and not $c/h$.

Finally Exercise 1.2: $T(n)=2T(n/2)+n/\log_2n$ is $n(1+H_K)$, and $H_K\approx\ln K+0.5772$.

In [ ]:
T = solve(2, 2, lambda n: n / lg(n), 60)
for K in (10, 20, 40, 60):
    n = 2.0 ** K
    HK = sum(1 / i for i in range(1, K + 1))
    print(f"K={K:2d}  T(n)/n = {T[K]/n:8.4f}   1+H_K = {1+HK:8.4f}   ln ln n = {math.log(math.log(n)):6.3f}")

### P1.2 Merge sort

We count comparisons with a wrapper. The worst case $C(n)$ satisfies $C(n)=C(\lceil n/2\rceil)+C(\lfloor n/2\rfloor)+n-1$ (a merge of $p+q$ elements can be forced to take $p+q-1$ comparisons by interleaving), and has the closed form $n\lceil\log_2n\rceil-2^{\lceil\log_2n\rceil}+1$, which Chapter 2 proves.

In [ ]:
def merge_sort(xs, cnt):
    if len(xs) <= 1:
        return xs
    m = len(xs) // 2
    A, B = merge_sort(xs[:m], cnt), merge_sort(xs[m:], cnt)
    out, i, j = [], 0, 0
    while i < len(A) and j < len(B):
        cnt[0] += 1
        if A[i] < B[j]: out.append(A[i]); i += 1
        else: out.append(B[j]); j += 1
    return out + A[i:] + B[j:]

@lru_cache(None)
def Cw(n):
    return 0 if n <= 1 else Cw((n + 1) // 2) + Cw(n // 2) + n - 1

clog = lambda n: (n - 1).bit_length()                 # ceil(log2 n) for n >= 1
for n in range(1, 65):
    assert Cw(n) == n * clog(n) - (1 << clog(n)) + 1 <= n * clog(n)
    assert math.ceil(lg(math.factorial(n))) <= Cw(n)
rows = []
for n in (4, 8, 16, 32, 64, 1000):
    avg = 0
    for _ in range(200):
        c = [0]; xs = random.sample(range(10 ** 6), n); assert merge_sort(xs, c) == sorted(xs); avg += c[0] / 200
    rows.append((n, math.ceil(lg(math.factorial(n))), round(avg, 1), Cw(n), n * clog(n)))
print(f"{'n':>5} {'ceil lg n!':>10} {'average':>9} {'worst':>7} {'n ceil lg n':>12}")
for r in rows: print(f"{r[0]:5d} {r[1]:10d} {r[2]:9.1f} {r[3]:7d} {r[4]:12d}")

Merge sort's worst case is within a few percent of the information bound, and its average is even closer: on random inputs, merges rarely run to the end of both lists.

### P1.3 Amortised cost traces

In [ ]:
def counter_trace(m):
    v, tot, out = 0, 0, []
    for _ in range(m):
        tot += (v ^ (v + 1)).bit_length(); v += 1; out.append(tot)
    return out

class DynArray:
    def __init__(self, shrink=None):
        self.s, self.c, self.cost, self.shrink = 0, 0, 0, shrink
    def append(self):
        if self.s == self.c:
            self.cost += self.s; self.c = max(1, 2 * self.c)
        self.s += 1; self.cost += 1
    def pop(self):
        self.s -= 1; self.cost += 1
        if self.shrink == "half" and self.c > 1 and self.s <= self.c // 2:
            self.c //= 2; self.cost += self.s
        if self.shrink == "quarter" and self.c >= 4 and self.s <= self.c // 4:
            self.c //= 2; self.cost += self.s

m = 4096
cnt = counter_trace(m)
D = DynArray(); arr = []
for _ in range(m): D.append(); arr.append(D.cost)
assert all(cnt[i] <= 2 * (i + 1) and arr[i] <= 3 * (i + 1) for i in range(m))

def adversarial(rule, m):
    D = DynArray(rule)
    for _ in range(m // 2 + 1): D.append()       # 2049 elements: just past a power of two
    out = []
    for i in range(m // 2):
        if i % 2 == 0: D.pop()
        else: D.append()
        out.append(D.cost)
    return out
half, quarter = adversarial("half", m), adversarial("quarter", m)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(cnt, label="binary counter"); ax[0].plot(arr, label="dynamic array")
ax[0].plot([2 * (i + 1) for i in range(m)], "--", lw=0.8, label="2m"); ax[0].plot([3 * (i + 1) for i in range(m)], "--", lw=0.8, label="3m")
ax[0].legend(); ax[0].set_xlabel("operations")
ax[1].plot(half, label="halve at 1/2 (Exercise 1.6a)"); ax[1].plot(quarter, label="halve at 1/4 (Exercise 1.6b)")
ax[1].set_yscale("log"); ax[1].legend(); ax[1].set_xlabel("alternating pops and appends")
plt.tight_layout(); plt.show()
print("cost of the alternating phase:", half[-1] - half[0], "with the bad rule,", quarter[-1] - quarter[0], "with the good one")

With the bad rule every operation of the alternating phase copies about $2048$ elements; with the good one, the array is never resized during that phase.

The queue made of two stacks: a single dequeue can be expensive, the total never is.

In [ ]:
In, Out, ops, worst, tot = [], [], 0, 0, 0
for i in range(20000):
    if random.random() < 0.55:
        In.append(i); c = 1
    elif In or Out:
        c = 0
        if not Out:
            while In: Out.append(In.pop()); c += 2
        Out.pop(); c += 1
    else:
        continue
    ops += 1; tot += c; worst = max(worst, c)
print(f"{ops} operations, total cost {tot} <= 3*ops = {3*ops}, most expensive single operation {worst}")

CPython over-allocates lists too, with a growth factor of about $9/8$ for large lists rather than $2$: the amortised cost is still $O(1)$, with a larger constant and less wasted memory (redo the proof of Proposition 1.12 with growth factor $\gamma>1$ and $\Phi=\frac{\gamma}{\gamma-1}s-\frac{1}{\gamma-1}c$: the amortised cost becomes $1+\gamma/(\gamma-1)$, which is $10$ for $\gamma=9/8$).

In [ ]:
import sys
L, last, jumps = [], sys.getsizeof([]), []
for i in range(200000):
    L.append(i); sz = sys.getsizeof(L)
    if sz != last: jumps.append((len(L), sz)); last = sz
caps = [(n, (sz - sys.getsizeof([])) // 8) for n, sz in jumps]
print("first capacities:", caps[:12])
print("growth factors near the end:", [round(caps[i + 1][1] / caps[i][1], 3) for i in range(len(caps) - 5, len(caps) - 1)])

### P1.4 Horn clauses in linear time

A clause is a pair `(hyps, concl)`: `concl` is a variable, or `None` for a goal clause.

In [ ]:
from collections import deque

def horn_linear(nvars, clauses):
    cnt, occ, Q = [], [[] for _ in range(nvars)], deque()
    true = [False] * nvars
    for idx, (hyps, concl) in enumerate(clauses):
        cnt.append(len(hyps))
        if concl is not None:
            for v in hyps: occ[v].append(idx)
            if not hyps: Q.append(concl)
    while Q:
        v = Q.popleft()
        if true[v]: continue
        true[v] = True
        for idx in occ[v]:
            cnt[idx] -= 1
            if cnt[idx] == 0: Q.append(clauses[idx][1])
    ok = all(not all(true[v] for v in hyps) for hyps, concl in clauses if concl is None)
    return ok, true

def horn_naive(nvars, clauses):
    true = [False] * nvars; changed = True
    while changed:
        changed = False
        for hyps, concl in clauses:
            if concl is not None and not true[concl] and all(true[v] for v in hyps):
                true[concl] = True; changed = True; break      # restart the search, as in Tome C1
    ok = all(not all(true[v] for v in hyps) for hyps, concl in clauses if concl is None)
    return ok, true

def brute(nvars, clauses):
    for bits in itertools.product([False, True], repeat=nvars):
        if all((not all(bits[v] for v in h)) or (c is not None and bits[c]) for h, c in clauses):
            return True
    return False

def random_horn(n, m, goal_p=0.1):
    cl = []
    for _ in range(m):
        k = random.choice([0, 1, 1, 2, 2, 3])
        hyps = [random.randrange(n) for _ in range(k)]
        cl.append((hyps, None if random.random() < goal_p else random.randrange(n)))
    return cl

for _ in range(2000):
    n = random.randint(1, 8); cl = random_horn(n, random.randint(1, 12))
    a, b, c = horn_linear(n, cl), horn_naive(n, cl), brute(n, cl)
    assert a == b and a[0] == c
print("2000 random formulas: linear = naive = brute force")

To time them we need formulas where many rules fire in sequence: a chain $x_0$, $x_0\to x_1$, $x_1\to x_2,\dots$, written in reverse order so that the naive loop has to search far each time, plus random noise.

In [ ]:
def chain(n):
    cl = [([i], i + 1) for i in range(n - 1)][::-1] + [([], 0)]
    return n, cl
res = []
for n in (250, 500, 1000, 2000, 4000):
    nv, cl = chain(n)
    t0 = time.perf_counter(); horn_linear(nv, cl); t1 = time.perf_counter(); horn_naive(nv, cl); t2 = time.perf_counter()
    res.append((n, t1 - t0, t2 - t1))
    print(f"n={n:5d}  linear {1e3*(t1-t0):7.2f} ms   naive {1e3*(t2-t1):9.1f} ms")
print("doubling n multiplies the naive time by about", round(res[-1][2] / res[-2][2], 1), "and the linear time by about", round(res[-1][1] / res[-2][1], 1))

### P1.5 Adversaries

The adversary of Theorem 1.18 keeps actual values and the two status flags of each element. Algorithms receive a `less(i, j)` oracle; the adversary checks at the end that the output is consistent with its final values.

In [ ]:
class MaxMinAdversary:
    def __init__(self, n):
        self.val = list(range(n)); random.shuffle(self.val)
        self.lost = [False] * n; self.won = [False] * n; self.count = 0
    def _raise(self, i): self.val[i] = max(self.val) + 1
    def _lower(self, i): self.val[i] = min(self.val) - 1
    def less(self, i, j):                       # is x_i < x_j ?
        self.count += 1
        nl = lambda k: not self.lost[k]; nw = lambda k: not self.won[k]
        if nl(i) and self.lost[j]: self._raise(i)
        elif nl(j) and self.lost[i]: self._raise(j)
        elif nw(i) and self.won[j]: self._lower(i)
        elif nw(j) and self.won[i]: self._lower(j)
        w, l = (j, i) if self.val[i] < self.val[j] else (i, j)
        self.won[w] = True; self.lost[l] = True
        return w == j

def naive(n, less):
    mx = mn = 0
    for i in range(1, n):
        if less(mx, i): mx = i
        if less(i, mn): mn = i
    return mx, mn

def pairs_perm(idx, less):
    n = len(idx); W, Lo = [], []
    for k in range(0, n - 1, 2):
        a, b = idx[k], idx[k + 1]
        if less(a, b): Lo.append(a); W.append(b)
        else: Lo.append(b); W.append(a)
    if n % 2: W.append(idx[-1]); Lo.append(idx[-1])
    mx = W[0]
    for i in W[1:]:
        if less(mx, i): mx = i
    mn = Lo[0]
    for i in Lo[1:]:
        if less(i, mn): mn = i
    return mx, mn

pairs = lambda n, less: pairs_perm(list(range(n)), less)
def shuffled_pairs(n, less):                    # the same method, pairing a random permutation
    idx = list(range(n)); random.shuffle(idx)
    return pairs_perm(idx, less)

print(f"{'n':>4} {'bound':>6} {'naive':>6} {'pairs':>6} {'shuffled pairs':>15}")
for n in (2, 3, 4, 5, 10, 11, 50, 101):
    out = []
    for alg in (naive, pairs, shuffled_pairs):
        A = MaxMinAdversary(n); mx, mn = alg(n, A.less)
        assert A.val[mx] == max(A.val) and A.val[mn] == min(A.val)
        out.append(A.count)
    bound = math.ceil(3 * n / 2) - 2
    assert min(out) >= bound
    print(f"{n:4d} {bound:6d} {out[0]:6d} {out[1]:6d} {out[2]:15d}")

Every algorithm is forced to at least $\lceil 3n/2\rceil-2$, and the pairing algorithm meets the bound exactly. The naive algorithm pays $2n-2$.

Now the weight adversary of Theorem 1.19, against the tournament and against the naive two-pass method (find the maximum, then the maximum of the rest).

In [ ]:
class SecondAdversary:
    def __init__(self, n):
        self.val = list(range(n)); random.shuffle(self.val)
        self.lost = [False] * n; self.w = [1] * n; self.count = 0
    def less(self, i, j):
        self.count += 1
        if not self.lost[i] and not self.lost[j]:
            win, lose = (i, j) if self.w[i] >= self.w[j] else (j, i)
            self.val[win] = max(self.val) + 1; self.w[win] += self.w[lose]; self.w[lose] = 0
        elif not self.lost[i] or not self.lost[j]:
            win = i if not self.lost[i] else j; lose = j if win == i else i
            self.val[win] = max(self.val) + 1
        else:
            win, lose = (i, j) if self.val[i] > self.val[j] else (j, i)
        self.lost[lose] = True
        return win == j

def two_pass(n, less):
    mx = 0
    for i in range(1, n):
        if less(mx, i): mx = i
    rest = [i for i in range(n) if i != mx]; s = rest[0]
    for i in rest[1:]:
        if less(s, i): s = i
    return mx, s

def tournament(n, less):
    beaten = {i: [] for i in range(n)}; players = list(range(n))
    while len(players) > 1:
        nxt = []
        for k in range(0, len(players) - 1, 2):
            a, b = players[k], players[k + 1]
            w, l = (b, a) if less(a, b) else (a, b)
            beaten[w].append(l); nxt.append(w)
        if len(players) % 2: nxt.append(players[-1])
        players = nxt
    ch = players[0]; cand = beaten[ch]; s = cand[0]
    for i in cand[1:]:
        if less(s, i): s = i
    return ch, s

print(f"{'n':>5} {'bound':>6} {'two-pass':>9} {'tournament':>11}")
for n in (2, 3, 8, 13, 64, 100, 1000):
    out = []
    for alg in (two_pass, tournament):
        A = SecondAdversary(n); mx, s = alg(n, A.less)
        srt = sorted(range(n), key=lambda i: A.val[i])
        assert mx == srt[-1] and s == srt[-2]
        out.append(A.count)
    bound = n + clog(n) - 2
    assert min(out) >= bound
    print(f"{n:5d} {bound:6d} {out[0]:9d} {out[1]:11d}")

For Lewis Carroll's $64$ players the tournament needs $68$ matches, and the adversary shows no schedule can do with fewer.

### P1.6 Merge insertion

Ford and Johnson: compare the elements in pairs; sort the larger elements of the pairs recursively (by merge insertion); the smaller partner $b_1$ of the smallest large element goes in front for free; then insert the other $b_i$ by binary insertion, in the order $b_3,b_2,\;b_5,b_4,\;b_{11},\dots,b_6,\;b_{21},\dots$ (the group ends are the Jacobsthal numbers $(2^{k+1}+(-1)^k)/3$), each one only into the part of the chain lying below its partner $a_i$. The order is chosen so that each insertion is into a chain of length $2^k-1$, where binary insertion is most efficient.

In [ ]:
def binary_insert(chain, x, hi, less):
    lo = 0
    while lo < hi:
        mid = (lo + hi) // 2
        if less(x, chain[mid]): hi = mid
        else: lo = mid + 1
    chain.insert(lo, x)

def jacobsthal_order(m):
    # order of insertion of b_2..b_m (1-indexed), group ends t_k = 3, 5, 11, 21, ...
    order, prev, k = [], 1, 2
    while prev < m:
        t = min((2 ** (k + 1) + (-1) ** k) // 3, m)
        order += list(range(t, prev, -1)); prev = t; k += 1
    return order

def merge_insertion(items, less):
    n = len(items)
    if n <= 1: return list(items)
    pairs, odd = [], (items[-1] if n % 2 else None)
    for k in range(0, n - 1, 2):
        x, y = items[k], items[k + 1]
        pairs.append((y, x) if less(x, y) else (x, y))       # (larger, smaller)
    big_sorted = merge_insertion(pairs, lambda p, q: less(p[0], q[0]))
    a = [p[0] for p in big_sorted]; b = [p[1] for p in big_sorted]
    if odd is not None: b.append(odd)
    chain = [b[0]] + a
    pos_a = list(range(1, len(a) + 1))                        # current positions of the a_i in the chain
    for i in jacobsthal_order(len(b)):
        x = b[i - 1]
        hi = pos_a[i - 1] if i <= len(a) else len(chain)      # insert below the partner a_i
        binary_insert(chain, x, hi, less)
        p = chain.index(x)
        pos_a = [q + 1 if q >= p else q for q in pos_a]
    return chain

def count_sort(alg, xs):
    c = [0]
    def less(u, v):
        c[0] += 1; return u < v
    out = alg(list(xs), less)
    assert out == sorted(xs)
    return c[0]

F = lambda n: sum(math.ceil(lg(3 * k / 4)) for k in range(1, n + 1))
print(f"{'n':>3} {'ceil lg n!':>10} {'F(n)':>5} {'worst over all n!':>18} {'merge sort':>11}")
for n in range(1, 9):
    worst = max(count_sort(merge_insertion, p) for p in itertools.permutations(range(n)))
    assert worst == F(n)
    print(f"{n:3d} {math.ceil(lg(math.factorial(n))):10d} {F(n):5d} {worst:18d} {Cw(n):11d}")

The exhaustive worst case equals Ford and Johnson's formula and the information bound for every $n\le 8$, and for $n=5$ it is $7$ (Exercise 1.8). The first gap, $F(12)=30>29$, is real: Wells showed in 1965 that no algorithm sorts $12$ elements in $29$ comparisons. On larger $n$ we can only sample orderings, so the maximum below is a lower estimate of the worst case.

In [ ]:
for n in (12, 20, 50, 100):
    w = max(count_sort(merge_insertion, random.sample(range(n), n)) for _ in range(300))
    print(f"n={n:3d}  ceil lg n! = {math.ceil(lg(math.factorial(n))):4d}   F(n) = {F(n):4d}   largest seen = {w:4d}   merge sort worst = {Cw(n):4d}")

### P1.7 Euclid, counted

In [ ]:
def steps(a, b):
    k = 0
    while b: a, b = b, a % b; k += 1
    return k

fib = [0, 1]
while len(fib) < 60: fib.append(fib[-1] + fib[-2])
best = {}
for b in range(1, 10 ** 4):
    for a in range(b + 1, 2 * b + 1):          # a mod b ranges over everything; a < 2b suffices for the worst case
        s = steps(a, b); d = len(str(b))
        if s > best.get(d, (0,))[0]: best[d] = (s, a, b)
for d, (s, a, b) in sorted(best.items()):
    print(f"{d} digits: {s} divisions for (a,b)=({a},{b}); Fibonacci pair: {a in fib and b in fib}; 5*digits = {5*d}")

For each number of digits the worst pair is a pair of consecutive Fibonacci numbers (restricting to $a<2b$ loses nothing: replacing $a$ by $b+(a\bmod b)$ lowers the first quotient to $1$ and does not change the number of divisions). Lamé's $5\times$digits is generous; the true constant is $\log_\varphi 10\approx4.785$.

Now the bit cost: schoolbook division of an $m$-bit by an $\ell$-bit number costs $\ell(m-\ell+1)$ in our model.

In [ ]:
bits = lambda x: x.bit_length()
def euclid_bitcost(a, b):
    cost = 0
    while b:
        cost += bits(b) * (bits(a) - bits(b) + 1); a, b = b, a % b
    return cost
data = []
for la in (64, 128, 256, 512, 1024, 2048):
    for lb in (la // 4, la // 2, la):
        a = random.getrandbits(la) | (1 << (la - 1)); b = random.getrandbits(lb) | (1 << (lb - 1))
        if b >= a: a, b = b + a, a
        data.append((la, lb, euclid_bitcost(a, b) / (bits(a) * bits(b))))
for la, lb, r in data[-9:]:
    print(f"l(a)={la:5d} l(b)={lb:5d}  cost / (l(a) l(b)) = {r:.3f}")
fib_r = [euclid_bitcost(fib[k + 1], fib[k]) / (bits(fib[k + 1]) * bits(fib[k])) for k in range(10, 59)]
print("Fibonacci pairs, ratio at the end:", round(fib_r[-1], 3))

The ratio stays bounded, as Proposition 1.3 promises: Euclid's algorithm costs about as much as one schoolbook multiplication, whatever the sizes. Chapter 10 will push this further, to the extended algorithm and to fast gcd.